# 解説用ノート — 記事の図を実データから生成

`build_pro_onnx_by_persona.ipynb`（学習ノート）は**触りません**。学習の成果物とマップ生成ロジックから、記事 `how-agents-see-think-move.md` に貼る図だけを生成します。

- 上から順に実行 → `figures/` に PNG が出力されます。
- fig6(DINOv2) だけ GPU 推奨。他は CPU で数秒です。
- テクスチャは `TEX_DIR`（学習と同じ Drive フォルダ）を読み、無ければ単色で代替します。

In [ ]:
# ── セットアップ: 定数・マップ生成・経路探索・CPUレイキャスタ ──
import numpy as np, json, os, random, math
from collections import deque
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

# Google Drive をマウント (Colab のみ。ローカルでは無視される)
try:
    from google.colab import drive; drive.mount("/content/drive")
    print("Drive mounted")
except Exception as e:
    print("Drive mount skipped:", e)

TEX_DIR   = "/content/drive/MyDrive/mesa_textures"   # 学習と同じ実写テクスチャ (無ければ単色代替)
REWARD_JSON = "/content/drive/MyDrive/mesa_out/persona_rewards.json"  # cell12 の書き出し先に合わせる

# 出力先: Drive がマウントされていれば Drive/mesa_figures に保存 (mesa_textures の隣・永続)。
# 無ければローカル ./figures (Colab の /content は一時領域なのでセッション終了で消える点に注意)。
_DRIVE="/content/drive/MyDrive"
OUT = f"{_DRIVE}/mesa_figures" if os.path.isdir(_DRIVE) else "figures"
os.makedirs(OUT, exist_ok=True)
print("★ 図の出力先:", os.path.abspath(OUT))

GRID=30; OTHER,ROAD,BUILDING,TREE=0,1,2,3
BLDG_TYPE_NAMES=["kiosk","conbini","pharmacy","cafe","gyudon","ramen","bento","shop","house","post",
 "bank","apartment","hotel","office","tower","supermarket","temple","school","station","library",
 "hospital","cityhall","museum","stadium","mall"]
FOOTPRINT=[1]*15+[2]*10
COLORS=["#d08030","#20a8e0","#30b070","#8B5E3C","#e8a020","#e03030","#20a020","#c060a0","#a06040","#d04040",
 "#808890","#9088a0","#c0a060","#4060a0","#6070b0","#40a060","#c04040","#e0b040","#7080a0","#8060a0",
 "#e0e0f0","#b0b4b8","#a09060","#60a080","#5878a0"]
FP1=[i for i,f in enumerate(FOOTPRINT) if f==1]; FP2=[i for i,f in enumerate(FOOTPRINT) if f==2]
ROLE={"A":"Explorer","B":"Homebody","C":"Social","D":"Businessman","E":"Tourist"}

def make_map(seed):
    rng=random.Random(seed); g=np.full((GRID,GRID),OTHER,np.int32)
    rows=list(range(0,GRID,4)); cols=list(range(0,GRID,4))
    for r in rows: g[r,:]=ROAD
    for c in cols: g[:,c]=ROAD
    for ri in range(len(rows)-1):
        for ci in range(len(cols)-1):
            r0,r1,c0,c1=rows[ri]+1,rows[ri+1],cols[ci]+1,cols[ci+1]
            cells=[(r,c) for r in range(r0,r1) for c in range(c0,c1)]
            if not cells: continue
            patch=None
            if r1-r0>=2 and c1-c0>=2 and rng.random()<0.42:
                pr=r0 if rng.random()<0.5 else r1-2; pc=c0 if rng.random()<0.5 else c1-2
                for r in range(pr,pr+2):
                    for c in range(pc,pc+2): g[r,c]=BUILDING
                patch=(pr,pc)
            b=rng.choice(cells); g[b[0],b[1]]=BUILDING
            for r,c in cells:
                if (r,c)==b: continue
                if patch and patch[0]<=r<patch[0]+2 and patch[1]<=c<patch[1]+2: continue
                v=rng.random()
                if v<0.25: g[r,c]=TREE
                elif v<0.45: g[r,c]=BUILDING
    for r in rows: g[r,:]=ROAD
    for c in cols: g[:,c]=ROAD
    return g

def assign_types(g, seed=123):
    rng=np.random.default_rng(seed); bt=np.full((GRID,GRID),-1,np.int64)
    bset={(r,c) for r in range(GRID) for c in range(GRID) if g[r,c]==BUILDING}; asg=set()
    for r in range(GRID-1):
        for c in range(GRID-1):
            if (r,c) in asg: continue
            quad=[(r,c),(r+1,c),(r,c+1),(r+1,c+1)]
            if all(q in bset for q in quad) and all(q not in asg for q in quad):
                ti=int(FP2[rng.integers(0,len(FP2))])
                for q in quad: asg.add(q); bt[q]=ti
    for (r,c) in bset:
        if (r,c) in asg: continue
        bt[r,c]=int(FP1[rng.integers(0,len(FP1))])
    return bt

def reachable(g):
    seen=np.zeros((GRID,GRID),bool); q=deque()
    for r in range(GRID):
        for c in range(GRID):
            if g[r,c]==ROAD: seen[r,c]=True; q.append((r,c))
    while q:
        r,c=q.popleft()
        for dr,dc in((-1,0),(1,0),(0,-1),(0,1)):
            nr,nc=r+dr,c+dc
            if 0<=nr<GRID and 0<=nc<GRID and not seen[nr,nc] and g[nr,nc] in (ROAD,BUILDING):
                seen[nr,nc]=True; q.append((nr,nc))
    return seen

def plan_path(g, s, t):  # 道路優先ダイクストラ (server.js planPath と一致)
    import heapq
    def ok(r,c): return 0<=r<GRID and 0<=c<GRID and g[r,c] in (ROAD,BUILDING)
    if not ok(*s) or not ok(*t): return None
    dist={s:0}; prev={}; pq=[(0,s)]
    while pq:
        d,u=heapq.heappop(pq)
        if u==t: break
        if d>dist.get(u,1e9): continue
        for dr,dc in((-1,0),(1,0),(0,-1),(0,1)):
            v=(u[0]+dr,u[1]+dc)
            if not ok(*v): continue
            nd=d+(1 if g[v]==ROAD else 6)
            if nd<dist.get(v,1e9): dist[v]=nd; prev[v]=u; heapq.heappush(pq,(nd,v))
    if t not in dist: return None
    path=[t]
    while path[-1]!=s: path.append(prev[path[-1]])
    return path[::-1]

# テクスチャ (64x64) を読む。無ければ単色。返り値 (25,64,64,3) float[0,1]
def load_tex(size=64):
    from glob import glob
    try: from PIL import Image
    except Exception: Image=None
    T=np.zeros((25,size,size,3),np.float32)
    for i,name in enumerate(BLDG_TYPE_NAMES):
        hit=None
        if Image is not None:
            for ext in ("jpg","jpeg","png"):
                h=glob(f"{TEX_DIR}/**/{name}.{ext}",recursive=True)
                if h: hit=h[0]; break
        if hit:
            T[i]=np.asarray(Image.open(hit).convert("RGB").resize((size,size)),np.float32)/255
        else:
            c=COLORS[i]; T[i]=np.array([int(c[1:3],16),int(c[3:5],16),int(c[5:7],16)],np.float32)/255
    return T

# CPU DDA レイキャスタ (学習の render_fp_batch と同じ式・1エージェント版)
def render_fpv(g, bt, TEX, x, y, th, W=220, H=140, FOV=math.radians(60)):
    img=np.zeros((H,W,3),np.float32)
    img[:H//2]=np.array([0.72,0.80,0.90]); img[H//2:]=np.array([0.42,0.45,0.40])
    dirx,diry=math.cos(th),math.sin(th)
    plane=math.tan(FOV/2); pxx,pyy=-diry*plane,dirx*plane
    for col in range(W):
        cam=2*col/W-1; rdx=dirx+pxx*cam; rdy=diry+pyy*cam
        mapx,mapy=int(x),int(y)
        ddx=1e30 if rdx==0 else abs(1/rdx); ddy=1e30 if rdy==0 else abs(1/rdy)
        if rdx<0: stepx=-1; sdx=(x-mapx)*ddx
        else: stepx=1; sdx=(mapx+1-x)*ddx
        if rdy<0: stepy=-1; sdy=(y-mapy)*ddy
        else: stepy=1; sdy=(mapy+1-y)*ddy
        hit=-1; side=0
        for _ in range(64):
            if sdx<sdy: sdx+=ddx; mapx+=stepx; side=0
            else: sdy+=ddy; mapy+=stepy; side=1
            if not(0<=mapx<GRID and 0<=mapy<GRID): break
            if g[mapx,mapy]==BUILDING: hit=int(bt[mapx,mapy]); break
        if hit<0: continue
        perp=(sdx-ddx) if side==0 else (sdy-ddy); perp=max(perp,1e-4)
        lineH=int(H/perp); ds=max(0,-lineH//2+H//2); de=min(H-1,lineH//2+H//2)
        wallX=(y+perp*rdy) if side==0 else (x+perp*rdx); wallX-=math.floor(wallX)
        texX=min(63,int(wallX*64)); t=TEX[hit%len(TEX)]
        br=min(1.0,max(0.35,1-perp/9))
        for yy in range(ds,de+1):
            texY=min(63,int((yy-ds)/max(1,lineH)*64)); img[yy,col]=t[texY,texX]*br
    return img

# 全図で共有する1枚のマップ (seed 固定で再現可能)
g=make_map(7); bt=assign_types(g); reach=reachable(g)
base=np.zeros((GRID,GRID,3))
for r in range(GRID):
    for c in range(GRID):
        t=g[r,c]
        base[r,c]=(0.86,0.88,0.90) if t==ROAD else (0.95,0.96,0.93) if t==OTHER else (0.55,0.78,0.45) if t==TREE else (1,1,1)
print("setup OK — helpers + shared map ready")

In [ ]:
# ── Fig 1: マップ + 到達可能/孤立建物 (g,bt,reach,base はセットアップで定義済み) ──
fig,ax=plt.subplots(figsize=(6.2,6.2)); ax.imshow(base,origin="upper"); iso=0
for r in range(GRID):
    for c in range(GRID):
        if g[r,c]==BUILDING:
            ax.add_patch(Rectangle((c-0.5,r-0.5),1,1,color=COLORS[bt[r,c]%25]))
            if not reach[r,c]: ax.plot(c,r,"x",color="red",ms=9,mew=2.5); iso+=1
ax.plot([],[],"x",color="red",ms=9,mew=2.5,ls="",label=f"isolated (excluded): {iso}")
ax.legend(loc="upper right",fontsize=8,framealpha=0.9)
ax.set_title("City map: buildings by type\nred X = isolated (excluded from spawn/goal)",fontsize=9)
ax.set_xticks([]); ax.set_yticks([]); plt.tight_layout()
plt.savefig(f"{OUT}/fig1_map_reachable.png",dpi=130); plt.show()

In [ ]:
# ── Fig 2: ペルソナ別 報酬パラメータ ヒートマップ ──
# persona_rewards.json を複数候補から探索。見つからなければ埋め込みデフォルトで描画。
import glob as _glob
_cands=[REWARD_JSON,"persona_rewards.json","/content/persona_rewards.json"]+_glob.glob("/content/drive/MyDrive/**/persona_rewards.json",recursive=True)
rw=None
for _p in _cands:
    if _p and os.path.exists(_p): rw=json.load(open(_p)); print("loaded rewards:",_p); break
if rw is None:
    print("persona_rewards.json が見つからないので埋め込みデフォルトを使用 (学習ノート cell12 と同じ内容)")
    rw={
     "A":{"persona_name":"Explorer","explore_bonus":4.5,"building_bonus":0.1,"road_bonus":0.4,"forward_bias":0.3,"revisit_penalty":1.8,"violation_penalty":3.0,"goal_reward":10.0,"step_penalty":0.05,"social_bonus":0.0,"stall_penalty":1.5,"food_bonus":0.0,"dwell_bonus":0.0,"social_dwell_bonus":0.0},
     "B":{"persona_name":"Homebody","explore_bonus":0.1,"building_bonus":0.5,"road_bonus":0.3,"forward_bias":0.6,"revisit_penalty":0.0,"violation_penalty":5.0,"goal_reward":28.0,"step_penalty":0.4,"social_bonus":0.0,"stall_penalty":2.0,"food_bonus":0.0,"dwell_bonus":1.2,"social_dwell_bonus":0.0},
     "C":{"persona_name":"Social","explore_bonus":0.5,"building_bonus":1.2,"road_bonus":0.2,"forward_bias":0.2,"revisit_penalty":0.0,"violation_penalty":3.0,"goal_reward":15.0,"step_penalty":0.08,"social_bonus":3.5,"stall_penalty":0.8,"food_bonus":0.0,"dwell_bonus":0.0,"social_dwell_bonus":2.0},
     "D":{"persona_name":"Businessman","explore_bonus":0.0,"building_bonus":0.3,"road_bonus":0.2,"forward_bias":0.9,"revisit_penalty":0.2,"violation_penalty":4.0,"goal_reward":30.0,"step_penalty":0.5,"social_bonus":0.0,"stall_penalty":2.0,"food_bonus":0.0,"dwell_bonus":0.0,"social_dwell_bonus":0.0},
     "E":{"persona_name":"Tourist","explore_bonus":2.0,"building_bonus":2.0,"road_bonus":0.1,"forward_bias":0.1,"revisit_penalty":0.3,"violation_penalty":2.5,"goal_reward":20.0,"step_penalty":0.02,"social_bonus":0.5,"stall_penalty":0.4,"food_bonus":0.0,"dwell_bonus":1.5,"social_dwell_bonus":0.0}}
pids=list(rw); axes=[k for k in rw[pids[0]] if isinstance(rw[pids[0]][k],(int,float))]
M=np.array([[rw[p].get(k,0) for k in axes] for p in pids],float)
Mn=M/(np.abs(M).max(axis=0)+1e-9)
fig,ax=plt.subplots(figsize=(max(7,len(axes)*0.7),3.2))
ax.imshow(Mn,cmap="RdYlGn",aspect="auto",vmin=-1,vmax=1)
ax.set_xticks(range(len(axes))); ax.set_xticklabels(axes,rotation=45,ha="right",fontsize=8)
ax.set_yticks(range(len(pids))); ax.set_yticklabels([f"{p}: {ROLE.get(p,p)}" for p in pids],fontsize=8)
for i in range(len(pids)):
    for j in range(len(axes)): ax.text(j,i,f"{M[i,j]:g}",ha="center",va="center",fontsize=6.5)
ax.set_title("Per-persona reward parameters (each character = different reward DNA)",fontsize=9)
plt.tight_layout(); plt.savefig(f"{OUT}/fig2_reward_heatmap.png",dpi=130); plt.show()

In [ ]:
# ── Fig 3: B モードの A* 経路例 ──
spawn=[(r,c) for r in range(GRID) for c in range(GRID) if g[r,c]==BUILDING and reach[r,c]]
s=random.Random(3).choice(spawn)
houses=[(r,c) for (r,c) in spawn if bt[r,c]==8] or spawn
t=min(houses,key=lambda b:(b[0]-15)**2+(b[1]-15)**2); path=plan_path(g,s,t)
fig,ax=plt.subplots(figsize=(6.2,6.2)); ax.imshow(base,origin="upper")
for r in range(GRID):
    for c in range(GRID):
        if g[r,c]==BUILDING: ax.add_patch(Rectangle((c-0.5,r-0.5),1,1,color=COLORS[bt[r,c]%25],alpha=0.55))
if path: ax.plot([p[1] for p in path],[p[0] for p in path],"-",color="#1030c0",lw=2.5,label=f"A* route ({len(path)} cells)")
ax.plot(s[1],s[0],"o",color="#00a000",ms=12,label="start")
ax.plot(t[1],t[0],"*",color="#c00000",ms=18,label="goal: house")
ax.legend(loc="upper right",fontsize=8,framealpha=0.9)
ax.set_title("B mode (navigate): road-preferring A* route to the goal building",fontsize=9)
ax.set_xticks([]); ax.set_yticks([]); plt.tight_layout()
plt.savefig(f"{OUT}/fig3_route.png",dpi=130); plt.show()

In [ ]:
# ── Fig 4: 観測ベクトル 421 の内訳 ──
fig,ax=plt.subplots(figsize=(8,1.9)); x=0
segs=[("CLS",384,"#4060a0"),("z",25,"#e0b040"),("aux",12,"#40a060")]
for name,w,col in segs:
    ax.add_patch(Rectangle((x,0),w,1,color=col))
    ax.text(x+w/2,0.5,f"{name}\n{w}",ha="center",va="center",fontsize=9,color="white",fontweight="bold"); x+=w
ax.set_xlim(0,421); ax.set_ylim(-0.05,1); ax.set_yticks([])
ax.set_title("Policy input = 421 dims  (read every decision)",fontsize=11)
ax.set_xlabel("dimension index")
ax.text(421,-0.55,"CLS=DINOv2 vision / z=goal type one-hot(25) / aux=compass3+visited4+social2+obstacle3",
        ha="right",va="top",fontsize=7.5,color="#444")
plt.tight_layout(); plt.savefig(f"{OUT}/fig4_obs_vector.png",dpi=130,bbox_inches="tight"); plt.show()

In [ ]:
# ── Fig 5: FPV観測サンプル (エージェントが見ている一人称視点) ──
TEX=load_tex()
# 正面数マス以内に建物が来る道路セル/向きを選んでレンダリング (確実に街並みが写る)
rng=random.Random(11); views=[]
roadcells=[(r,c) for r in range(GRID) for c in range(GRID) if g[r,c]==ROAD]; rng.shuffle(roadcells)
for (r,c) in roadcells:
    for th in (0,math.pi/2,math.pi,3*math.pi/2):
        hit=False
        for d in range(1,6):
            rr=int(r+math.cos(th)*d); cc=int(c+math.sin(th)*d)
            if not(0<=rr<GRID and 0<=cc<GRID) or g[rr,cc]==TREE: break
            if g[rr,cc]==BUILDING: hit=True; break
        if hit: views.append(render_fpv(g,bt,TEX,r+0.5,c+0.5,th)); break
    if len(views)>=4: break
fig,axs=plt.subplots(1,max(1,len(views)),figsize=(3*max(1,len(views)),2))
if len(views)==1: axs=[axs]
for a,im in zip(axs,views): a.imshow(np.clip(im,0,1)); a.set_xticks([]); a.set_yticks([])
fig.suptitle("FPV observations (what the agent sees each step)",fontsize=10)
plt.tight_layout(); plt.savefig(f"{OUT}/fig5_fpv_samples.png",dpi=130); plt.show()

In [ ]:
# ── Fig 6: DINOv2 特徴のクラスタ (GPU推奨) ──
# 各建物タイプを正面に置いた合成FPVを DINOv2 に通し、cls を 2次元(PCA)に落として色分け。
# 専用分類器なしでもタイプが分離する＝「事前分類しない」の可視化。
import torch
TEX=load_tex()
dev="cuda" if torch.cuda.is_available() else "cpu"
dino=torch.hub.load("facebookresearch/dinov2","dinov2_vits14").to(dev).eval()
for p in dino.parameters(): p.requires_grad=False
mean=torch.tensor([0.485,0.456,0.406],device=dev).view(1,3,1,1)
std =torch.tensor([0.229,0.224,0.225],device=dev).view(1,3,1,1)

def fpv_for_type(ti, ang):
    # 空マップの正面(奥)に そのタイプの建物1マスを置いて正面を向く
    gg=np.full((GRID,GRID),ROAD,np.int32); bb=np.full((GRID,GRID),-1,np.int64)
    ax_,ay_=15.5,15.5
    tx,ty=int(ax_+math.cos(ang)*3),int(ay_+math.sin(ang)*3)
    tx=min(GRID-1,max(0,tx)); ty=min(GRID-1,max(0,ty))
    gg[tx,ty]=BUILDING; bb[tx,ty]=ti
    return render_fpv(gg,bb,TEX,ax_,ay_,ang,W=224,H=224)

imgs=[]; labels=[]
for ti in range(25):
    for ang in (0,math.pi/2,math.pi,3*math.pi/2, math.pi/4, 3*math.pi/4):
        imgs.append(fpv_for_type(ti,ang)); labels.append(ti)
X=torch.tensor(np.stack(imgs),dtype=torch.float32,device=dev).permute(0,3,1,2)
feats=[]
with torch.no_grad():
    for i in range(0,X.shape[0],64):
        xb=(X[i:i+64]-mean)/std
        feats.append(dino.forward_features(xb)["x_norm_clstoken"].float().cpu().numpy())
F=np.concatenate(feats,0); labels=np.array(labels)
# PCA (numpy SVD) で 2次元へ
Fc=F-F.mean(0); U,S,Vt=np.linalg.svd(Fc,full_matrices=False); P=Fc@Vt[:2].T
fig,ax=plt.subplots(figsize=(7,6))
for ti in range(25):
    m=labels==ti; ax.scatter(P[m,0],P[m,1],s=28,color=COLORS[ti],label=BLDG_TYPE_NAMES[ti],edgecolors="k",linewidths=0.3)
ax.legend(ncol=2,fontsize=6,loc="upper right")
ax.set_title("DINOv2 CLS features (PCA): building types cluster without any classifier",fontsize=9)
ax.set_xticks([]); ax.set_yticks([]); plt.tight_layout()
plt.savefig(f"{OUT}/fig6_dinov2_cluster.png",dpi=130); plt.show()

In [ ]:
# ── 生成された図の一覧と保存先 ──
print("図の保存先:", os.path.abspath(OUT))
for f in sorted(os.listdir(OUT)):
    if f.endswith(".png"): print("  ", os.path.join(OUT, f))
print("\nDrive に出ている場合は drive.google.com の MyDrive/mesa_figures から見られます。")